# Reproduce the paper on Google Colab

Runtime → Change runtime type → **GPU (T4 or better)**. The whole notebook takes ~1–2 h for 5 seeds.

Every result in the paper comes from the commands below; nothing is computed in the notebook itself.

In [ ]:
!git clone https://github.com/eseeyuh/pneumonia-xai-detector.git
%cd pneumonia-xai-detector
!pip install -q -e ".[research]"

## 1. Data

Download Kermany et al. (2018) from Kaggle. Create an API token on kaggle.com → Settings → *Create New Token*
and upload the `kaggle.json` it gives you when the cell asks.

In [ ]:
from google.colab import files
import os
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    files.upload()  # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data -q
!unzip -q -o data/chest-xray-pneumonia.zip -d data && ls data/chest_xray

## 2. How much would a random image-level split leak?

This number goes into the paper's *Methods* section.

In [ ]:
!python -m pxai.data --root data/chest_xray --audit

## 3. Patient-level split (official test set kept intact)

In [ ]:
!python -m pxai.data --root data/chest_xray --mode official --out data/splits/kermany.csv

## 4. Train 5 seeds (single model = seed 42, ensemble = all 5)

In [ ]:
for seed in [42, 1, 2, 3, 4]:
    !python -m pxai.train --config configs/default.yaml --set seed={seed} output_dir=runs/seed{seed} data.num_workers=2

## 5. Internal evaluation (operating point fitted on val, frozen for test)

In [ ]:
!python -m pxai.evaluate --manifest data/splits/kermany.csv --root data/chest_xray \
    --checkpoints runs/seed42/best.pt runs/seed1/best.pt runs/seed2/best.pt runs/seed3/best.pt runs/seed4/best.pt \
    --tta-samples 16 --workers 2 --out results/kermany

In [ ]:
import json, pandas as pd
m = json.load(open("results/kermany/metrics.json"))["test"]["single_model"]
print({k: m[k] for k in ("auc", "sensitivity", "specificity", "threshold")})
pd.DataFrame(m["selective"]).T.sort_values("aurc")

## 6. External validation on RSNA (adults)

Accept the competition rules on Kaggle first:
https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules

In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p data/rsna -q
!unzip -q -o data/rsna/rsna-pneumonia-detection-challenge.zip -d data/rsna
!python scripts/prepare_rsna.py --rsna-dir data/rsna --out-dir data/rsna_png
!python -m pxai.evaluate --manifest data/splits/rsna.csv --root . \
    --checkpoints runs/seed42/best.pt --operating-point results/kermany/operating_point.json \
    --workers 2 --out results/rsna
!python scripts/eval_saliency.py --manifest data/splits/rsna.csv --boxes data/splits/rsna_boxes.csv \
    --checkpoint runs/seed42/best.pt --out results/saliency

## 7. Save everything to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/pneumonia-xai
!cp -r runs results data/splits /content/drive/MyDrive/pneumonia-xai/